In [1]:
# 구글드라이브 연동 그리고 깃허브 클론/풀
import os
from google.colab import drive

drive.mount('/content/drive')

%cd /content
if os.path.exists('/content/korean-chatbot'):
    %cd korean-chatbot
    !git pull
else:
    !git clone https://github.com/kkkk2058/korean-chatbot.git
    %cd korean-chatbot

!pip install -r requirements.txt

Mounted at /content/drive
/content
Cloning into 'korean-chatbot'...
remote: Enumerating objects: 74, done.
remote: Counting objects: 100% (74/74), done.
remote: Compressing objects: 100% (53/53), done.
remote: Total 74 (delta 37), reused 53 (delta 19), pack-reused 0 (from 0)
Receiving objects: 100% (74/74), 17.11 KiB | 5.70 MiB/s, done.
Resolving deltas: 100% (37/37), done.
/content/korean-chatbot


In [2]:
# 허깅페이스에서 데이터 가져오기
from datasets import load_dataset

# streaming=True 필수!
ds = load_dataset("heegyu/namuwiki-extracted", split="train", streaming=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/1.48k [00:00<?, ?B/s]

In [3]:
import re
import os
from datasets import load_dataset

ds = load_dataset("heegyu/namuwiki-extracted", split="train", streaming=True)

MAX_SAMPLES = 100_000

def clean(text):
    # ① [수정] 대괄호는 지우고 알맹이 글자만 남기기 (예: [[조선]] -> 조선)
    text = re.sub(r'\[\[(?:[^\]|]*\|)?([^\]]+)\]\]', r'\1', text)

    # ② URL 제거
    text = re.sub(r'https?://\S+', '', text)

    # ③ ReDoS 방지를 위해 안전한 패턴으로 문단 구분선(== 제목 ==) 제거
    text = re.sub(r'==[^=\n]+==', '', text)

    # ④ 연속된 공백을 하나로 축소
    text = re.sub(r'\s+', ' ', text)

    # ⑤ 만에 하나 짝이 안 맞아 남은 유령 대괄호('[', ']') 최종 청소
    text = text.replace('[', '').replace(']', '')

    return text.strip()

os.makedirs('data', exist_ok=True)

print("데이터 추출 및 정제 시작...")

buffer = []
saved_count = 0

with open('data/namuwiki.txt', 'w', encoding='utf-8') as f:
    for i, row in enumerate(ds):
        if saved_count >= MAX_SAMPLES:
            break

        text = clean(row['text'])

        if len(text) > 10:
            buffer.append(text + '\n')
            saved_count += 1

        # 1,000개씩 모아서 디스크에 한 번에 쓰기 (속도 향상)
        if len(buffer) >= 1000:
            f.writelines(buffer)
            buffer = []
            print(f"현재 {saved_count}개 저장 완료...")

    # 남은 버퍼 비우기
    if buffer:
        f.writelines(buffer)

print(f"완료! 총 {saved_count}개의 데이터가 저장되었습니다.")

데이터 추출 및 정제 시작...
현재 1000개 저장 완료...
현재 2000개 저장 완료...
현재 3000개 저장 완료...
현재 4000개 저장 완료...
현재 5000개 저장 완료...
현재 6000개 저장 완료...
현재 7000개 저장 완료...
현재 8000개 저장 완료...
현재 9000개 저장 완료...
현재 10000개 저장 완료...
현재 11000개 저장 완료...
현재 12000개 저장 완료...
현재 13000개 저장 완료...
현재 14000개 저장 완료...
현재 15000개 저장 완료...
현재 16000개 저장 완료...
현재 17000개 저장 완료...
현재 18000개 저장 완료...
현재 19000개 저장 완료...
현재 20000개 저장 완료...
현재 21000개 저장 완료...
현재 22000개 저장 완료...
현재 23000개 저장 완료...
현재 24000개 저장 완료...
현재 25000개 저장 완료...
현재 26000개 저장 완료...
현재 27000개 저장 완료...
현재 28000개 저장 완료...
현재 29000개 저장 완료...
현재 30000개 저장 완료...
현재 31000개 저장 완료...
현재 32000개 저장 완료...
현재 33000개 저장 완료...
현재 34000개 저장 완료...
현재 35000개 저장 완료...
현재 36000개 저장 완료...
현재 37000개 저장 완료...
현재 38000개 저장 완료...
현재 39000개 저장 완료...
현재 40000개 저장 완료...
현재 41000개 저장 완료...
현재 42000개 저장 완료...
현재 43000개 저장 완료...
현재 44000개 저장 완료...
현재 45000개 저장 완료...
현재 46000개 저장 완료...
현재 47000개 저장 완료...
현재 48000개 저장 완료...
현재 49000개 저장 완료...
현재 50000개 저장 완료...
현재 51000개 저장 완료...
현재 52000개 저장 완료...
현재 

In [3]:
# 마지막에 추가
import shutil

shutil.copy('data/namuwiki.txt', '/content/drive/MyDrive/korean-chatbot/data/namuwiki.txt')
print("구글 드라이브 저장 완료!")